# Optimization

Use {py:class}`liesel.optim.LieselOptim` to fit model parameters or find starting
values for MCMC. It minimizes the negative log posterior: likelihood plus priors.
Without priors, this gives a maximum likelihood fit.

For a small Normal model, a full-data fit takes:

In [1]:
import logging

import jax.numpy as jnp
import tensorflow_probability.substrates.jax.distributions as tfd

import liesel.model as lsl
import liesel.optim as opt

logging.getLogger("liesel").setLevel(logging.WARNING)

loc = lsl.Var.new_param(0.0, name="loc")
y = lsl.Var.new_obs(
    jnp.array([1.0, 2.0, 3.0]),
    dist=lsl.Dist(tfd.Normal, loc, 1.0),
    name="y",
)
model = lsl.Model(y)

Fit its mean with L-BFGS:

In [2]:
optim = opt.LieselOptim(
    model,
    optimizers="lbfgs",
    loss_monitor="train_full_data",
    show_progress=False,
)

result = optim.fit()

In [3]:
result.position_min_monitor

{'loc': Array(2., dtype=float32, weak_type=True)}

The fitted mean is 2, the average of these three observations.

L-BFGS needs the full data and a deterministic loss. For minibatches, use Adam.
The `optimizers` argument is required. Pass a configured Optax transformation,
such as `optimizers=optax.adam(0.01)`, to optimize all parameters with it.
The fitted values are returned separately; fitting does not change your model.

After a joint MAP fit, {meth}`optim.loss.approximate_joint_posterior(result) <liesel.optim.NegLogProbLoss.approximate_joint_posterior>`
can construct a Gaussian approximation for the optimized parameters. It uses
full-training curvature and checks that the selected position is a stationary
point with positive curvature; a validation or EMA minimum may not qualify.
See {py:meth}`~liesel.optim.NegLogProbLoss.approximate_joint_posterior` for controls
and {doc}`optimizer-laplace` for a walkthrough of draws and uncertainty in a
hierarchical model.

## Start here

```{toctree}
:maxdepth: 1

Overview <self>
tutorials/notebooks/09-liesel-optim-basic
Fit two data groups <tutorials/notebooks/10-liesel-optim-advanced>
```

## Common tasks

```{toctree}
:maxdepth: 1

optimizer-monitoring
optimizer-customization
optimizer-laplace
optimizer-splitting
optimizer-batching
optimizer-loss-scaling
optimizer-weighted-batching
optimizer-checkpointing
optimizer-migration
```

For arguments and defaults, see the {ref}`optimizer-api`.
{func}`goose.optim_flat <liesel.goose.optim_flat>` is deprecated and will be removed in 0.8.0; use the migration guide
above to update existing code.